In [40]:
import pandas as pd
import requests
import numpy
from dotenv import load_dotenv
import os

Creating a list that includes the top 200 rated movies on themoviedb.org using an API key

In [ ]:
load_dotenv()

api = os.environ.get("api_key")
url = "https://api.themoviedb.org/3/movie/top_rated"

top_movies = []
for page in range(1, 14):
    params = {
        "api_key": api,
        "language": "en-US",
        "page": page
    }
    
    response = requests.get(url, params=params)
    movies = response.json()
    
    if "results" in movies:
        top_movies.extend(movies["results"])

top_200 = top_movies[:200]

    

In [ ]:
movies

{'page': 13,
 'results': [{'adult': False,
   'backdrop_path': '/y2FNZLHynsevBnVTocGW6c3pQpr.jpg',
   'genre_ids': [18],
   'id': 422,
   'title': '8½',
   'original_language': 'it',
   'original_title': '8½',
   'overview': 'Guido Anselmi, a film director, finds himself creatively barren at the peak of his career. Urged by his doctors to rest, Anselmi heads for a luxurious resort, but a sorry group gathers—his producer, staff, actors, wife, mistress, and relatives—each one begging him to get on with the show. In retreat from their dependency, he fantasizes about past women and dreams of his childhood.',
   'popularity': 6.4544,
   'poster_path': '/23VIf7zNxAgaXwIlvzd3IvOd9sY.jpg',
   'release_date': '1963-02-14',
   'softcore': False,
   'video': False,
   'vote_average': 8.069,
   'vote_count': 2623},
  {'adult': False,
   'backdrop_path': '/9WoLTsRgrupkvCgs3bhsl2EjLhs.jpg',
   'genre_ids': [28, 12, 14],
   'id': 791373,
   'title': "Zack Snyder's Justice League",
   'original_langua

Getting the genre mapping from the website (ex. 16 for its genre ID is Animation) and creating genre dictionary to map the genre ID to an actual genre name.

In [22]:
genre_url = "https://api.themoviedb.org/3/genre/movie/list"
params = {
    "api_key": api,
    "language": "en-US"
}

response_genre = requests.get(genre_url, params=params)
genre_data = response_genre.json()
genre_dict = {str(genre["id"]): genre["name"] for genre in genre_data.get("genres", [])}

{'genres': [{'id': 28, 'name': 'Action'},
  {'id': 12, 'name': 'Adventure'},
  {'id': 16, 'name': 'Animation'},
  {'id': 35, 'name': 'Comedy'},
  {'id': 80, 'name': 'Crime'},
  {'id': 99, 'name': 'Documentary'},
  {'id': 18, 'name': 'Drama'},
  {'id': 10751, 'name': 'Family'},
  {'id': 14, 'name': 'Fantasy'},
  {'id': 36, 'name': 'History'},
  {'id': 27, 'name': 'Horror'},
  {'id': 10402, 'name': 'Music'},
  {'id': 9648, 'name': 'Mystery'},
  {'id': 10749, 'name': 'Romance'},
  {'id': 878, 'name': 'Science Fiction'},
  {'id': 10770, 'name': 'TV Movie'},
  {'id': 53, 'name': 'Thriller'},
  {'id': 10752, 'name': 'War'},
  {'id': 37, 'name': 'Western'}]}

Creating the raw dataset, adding a genre column, and saving it under the "raw" folder

In [46]:
top_200_movies = pd.DataFrame(top_200)
top_200_movies["genre_ids"] = top_200_movies["genre_ids"].str[0].astype(str)
top_200_movies["genre"] = top_200_movies["genre_ids"].map(genre_dict)
top_200_movies.to_csv("raw/top_200_raw.csv", index=False)

Creating a new, clean dataset that includes five variables that we want to focus on. Also saving the cleaned dataset in the "processed" folder.

In [48]:
top_200_movies_clean = top_200_movies[["title", "release_date", "overview", "genre", "vote_average"]].copy()
top_200_movies_clean["release_date"] = pd.to_datetime(top_200_movies_clean["release_date"])
top_200_movies_clean["release_year"] = top_200_movies_clean["release_date"].dt.year
top_200_movies_clean = top_200_movies_clean[["title", "release_year", "overview", "genre", "vote_average"]]
top_200_movies_clean.to_csv("processed/top_200_clean.csv", index=False)

# Exploratory Data Analysis

In [ ]:
# Top 5 highest rated movies
top_200_movies_clean.sort_values(by="vote_average", ascending=False).head()

,title,release_date,overview,genre,vote_average
0,Batman: Knightfall Part 1: Knightfall,2026-06-23,"Arkham Asylum has been destroyed, and all its ...",Animation,9.086
1,Avatar Aang: The Last Airbender,2026-07-24,"Avatar Aang, the world's last Airbender, learn...",Animation,9.000
2,Swapped,2026-05-01,"A small woodland creature and a majestic bird,...",Adventure,8.841
3,Accidental Partners,2026-03-12,Two women discover they were both scammed by t...,Comedy,8.819
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,2025-07-18,The Demon Slayer Corps are drawn into the Infi...,Animation,8.801


In [29]:
# Which genre dominates the dataset?
top_200_movies_clean.value_counts("genre")

genre
Drama              67
Animation          37
Comedy             19
Action             16
Adventure          12
Crime               8
Romance             8
Horror              6
Science Fiction     5
Family              5
Music               4
Thriller            4
Fantasy             3
War                 3
Mystery             2
Western             1
Name: count, dtype: int64

In [50]:
# Which year has the most movies in this dataset?
top_200_movies_clean.value_counts("release_year")

release_year
2026    21
2019    16
2020    12
2018     7
2025     6
        ..
1921     1
1958     1
1987     1
1996     1
1976     1
Name: count, Length: 70, dtype: int64

In [39]:
# Average rating by genre
top_200_movies_clean.groupby("genre")["vote_average"].mean().sort_values(ascending=False)

genre
Western            8.468000
Adventure          8.384333
Fantasy            8.343667
Action             8.326750
Science Fiction    8.325400
Animation          8.323757
Comedy             8.309737
Crime              8.308375
Thriller           8.305750
Music              8.288000
Drama              8.277537
Romance            8.277250
Family             8.260800
Horror             8.236000
War                8.229667
Mystery            8.152000
Name: vote_average, dtype: float64

### Data Attribution

This project uses the TMDB API but is not endorsed or certified by TMDB.

![TMDB Logo](https://www.themoviedb.org/assets/v4/logos/v2/blue_square_2-d537fb228cf3ded904ef09b136fe3fec72548ebc1fea3fbbd1ad9e36364db38b.svg)
